# G-nomix 
Local ancestry inference using pretrained model

In [ ]:
import os
from concurrent.futures import ThreadPoolExecutor
import subprocess
import pandas as pd
import polars as pl

In [ ]:
la_model_path = '/path/to/home/output/LA' #model_chm_22.pkl
la_snps_path = '/path/to/home/output/LA' #model_chm_22.snplist
work_dir = '/path/to/home/DATA/IMPUTED'

In [ ]:
chromosomes = list(range(1,23))

## Get overlapping variants for maximum match

TYPED only file

In [ ]:
inputvcf = f'{work_dir}/ALLCHR_OnlyTyped.vcf.gz'

Ensure chr{c} if the actual chr encoding is chr in vcf.gz

In [ ]:
def get_overlap(c):
    inputlist = f'{la_snps_path}/model_chm_{c}.snplist'
    outputvcf = f'{work_dir}/ALLCHR_OnlyTyped.{c}.vcf.gz'
    res = subprocess.run(
        [
            'bcftools', 'view',
            '-r', f'chr{c}',
            '-i', f'ID=@{inputlist}',
            inputvcf,
            '-Oz', '-o', outputvcf
        ],
        capture_output=True, text=True, check=False
    )
    if res.returncode != 0:
        print(f"[chr{c}] FAILED (rc={res.returncode}):\n{res.stderr}")
    elif res.stderr:
        print(f"[chr{c}] warnings:\n{res.stderr}")
    else:
        print(f"[chr{c}] OK")
    res.check_returncode() 

In [ ]:
with ThreadPoolExecutor(max_workers=8) as ex:
    list(ex.map(get_overlap, chromosomes))

## Print gnomix commands for terminal

In [ ]:
la_model_path = '/path/to/home/output/LA'
best_config = '/path/to/tools/Gnomix/gnomix/config.yaml'
la_dir = f'{work_dir}/LA_inference_results'
os.makedirs(la_dir, exist_ok=True)

In [ ]:
#!/usr/bin/env bash

work_dir="/path/to/home/DATA/IMPUTED"
la_dir="/path/to/home/DATA/IMPUTED/LA_inference_results"
la_model_path="/path/to/home/output/LA"
best_config="/path/to/tools/Gnomix/gnomix/config.yaml"

for c in {1..22}; do
  mkdir -p "${la_dir}/model_chm${c}"
  echo "chr${c} start $(date)"
  gnomix \
    "${work_dir}/ALLCHR_OnlyTyped.${c}.vcf.gz" \
    "${la_dir}/model_chm${c}" \
    "chr${c}" \
    False \
    "${la_model_path}/model_chm_${c}.pkl" \
    > "${la_dir}/gnomix_chr${c}.log" 2>&1
  echo "chr${c} done $(date)"
done

In [ ]:
nohup bash script.sh > gnomix_master.log 2>&1 &

In [ ]:
def run_gnomix(c):
    la_model_pkl = f'{la_model_path}/model_chm_{c}.pkl'
    inputvcf = f'{work_dir}/ALLCHR_OnlyTyped.{c}.vcf.gz'
    outputname = f'{la_dir}/model_chm{c}'
    logfile = f'{la_dir}/gnomix_chr{c}.log'
    
    cmd = f'gnomix {inputvcf} {outputname} chr{c} False {la_model_pkl} > {logfile} 2>&1 &'
    # print(f'\n=== Chromosome {c} ===')
    print(cmd)

for c in chromosomes:
    run_gnomix(c)

## Extract tracts from MSP file.

In [ ]:
! for c in {1..22}; do  python3 ~/TRACTOR-MIX/Tractor-Mix/extract_tracts.py --vcf ALLCHR_OnlyTyped.${c}.vcf.gz --msp LA_inference_results/model_chm${c}/MSP_${c}.msp --num-ancs 6 ; done